In [1]:
import re

documents = {
    "D1": "Men sun'iy intellekt texnologiyalarini o'rganaman",
    "D2": "Sun'iy intellekt zamonaviy texnologiya hisoblanadi",
    "D3": "Men intellekt haqida va sun'iy texnologiyalarni o'rganaman",
    "D4": "Axborot qidiruvi va ma'lumotlarni tahlil qilish"
}

print("Hujjatlar soni:", len(documents))

for doc_id, text in documents.items():
    print(doc_id, ":", text)

Hujjatlar soni: 4
D1 : Men sun'iy intellekt texnologiyalarini o'rganaman
D2 : Sun'iy intellekt zamonaviy texnologiya hisoblanadi
D3 : Men intellekt haqida va sun'iy texnologiyalarni o'rganaman
D4 : Axborot qidiruvi va ma'lumotlarni tahlil qilish


In [2]:
# 1. Tokenizatsiya

def tokenize(text):
    return re.findall(r"\b[\w']+\b", text.lower())

for doc_id, text in documents.items():
    print(doc_id, ":", tokenize(text))

D1 : ['men', "sun'iy", 'intellekt', 'texnologiyalarini', "o'rganaman"]
D2 : ["sun'iy", 'intellekt', 'zamonaviy', 'texnologiya', 'hisoblanadi']
D3 : ['men', 'intellekt', 'haqida', 'va', "sun'iy", 'texnologiyalarni', "o'rganaman"]
D4 : ['axborot', 'qidiruvi', 'va', "ma'lumotlarni", 'tahlil', 'qilish']


In [8]:
# 2. Pozitsion indeksni qurish

def build_positional_index(documents):
    index = {}

    for doc_id, text in documents.items():
        words = tokenize(text)

        for position, word in enumerate(words):
            if word not in index:
                index[word] = {}

            if doc_id not in index[word]:
                index[word][doc_id] = []

            index[word][doc_id].append(position)

    return index

index = build_positional_index(documents)

for word, postings in index.items():
    print(word, ":", postings)

men : {'D1': [0], 'D3': [0]}
sun'iy : {'D1': [1], 'D2': [0], 'D3': [4]}
intellekt : {'D1': [2], 'D2': [1], 'D3': [1]}
texnologiyalarini : {'D1': [3]}
o'rganaman : {'D1': [4], 'D3': [6]}
zamonaviy : {'D2': [2]}
texnologiya : {'D2': [3]}
hisoblanadi : {'D2': [4]}
haqida : {'D3': [2]}
va : {'D3': [3], 'D4': [2]}
texnologiyalarni : {'D3': [5]}
axborot : {'D4': [0]}
qidiruvi : {'D4': [1]}
ma'lumotlarni : {'D4': [3]}
tahlil : {'D4': [4]}
qilish : {'D4': [5]}


In [9]:
# 3. Phrase Query algoritmi

def phrase_query(terms, index):
    lists = [index.get(term, {}) for term in terms]

    if any(not posting for posting in lists):
        return []

    candidate_docs = set(lists[0])

    for posting in lists[1:]:
        candidate_docs &= set(posting)

    results = []

    for doc in candidate_docs:
        for p in lists[0][doc]:
            matched = True

            for i in range(1, len(terms)):
                if p + i not in lists[i][doc]:
                    matched = False
                    break

            if matched:
                results.append(doc)
                break

    return sorted(results)

phrase_query(["sun'iy", "intellekt"], index)

['D1', 'D2']

In [6]:
# 4. Proximity Query algoritmi

def proximity_query(termA, termB, max_distance, ordered, index):
    listA = index.get(termA, {})
    listB = index.get(termB, {})

    candidate_docs = set(listA) & set(listB)

    results = []

    for doc in candidate_docs:
        A = listA[doc]
        B = listB[doc]

        i = 0
        j = 0

        while i < len(A) and j < len(B):
            delta = B[j] - A[i]

            if ordered:
                if 0 < delta <= max_distance:
                    results.append(doc)
                    break

                if delta <= 0:
                    j += 1
                else:
                    i += 1

            else:
                if 0 < abs(delta) <= max_distance:
                    results.append(doc)
                    break

                if A[i] < B[j]:
                    i += 1
                else:
                    j += 1

    return sorted(results)

proximity_query(
    "sun'iy", "intellekt",
    max_distance=3,
    ordered=False,
    index=index
)

['D1', 'D2', 'D3']

In [7]:
# 5. Natijalarni taqqoslash

phrase_results = phrase_query(
    ["sun'iy", "intellekt"], index
)

proximity_results = proximity_query(
    "sun'iy", "intellekt",
    max_distance=3,
    ordered=False,
    index=index
)

print("PHRASE QUERY NATIJALARI:")
print(phrase_results)

print("\nPROXIMITY QUERY NATIJALARI:")
print(proximity_results)

print("\nTAQQOSLASH:")

for doc in documents:
    print(
        doc,
        "| Phrase:", doc in phrase_results,
        "| Proximity:", doc in proximity_results
    )

PHRASE QUERY NATIJALARI:
['D1', 'D2']

PROXIMITY QUERY NATIJALARI:
['D1', 'D2', 'D3']

TAQQOSLASH:
D1 | Phrase: True | Proximity: True
D2 | Phrase: True | Proximity: True
D3 | Phrase: False | Proximity: True
D4 | Phrase: False | Proximity: False
